In [1]:
from src.modules.decoder.decoder import Decoder
from src.modules.data.data_ingestion import DataIngestion

In [2]:
data_ingestion = DataIngestion()
data = data_ingestion.load_data()

In [3]:
split_idx = int(len(data) * 0.8)

train_text = data[:split_idx]
val_text = data[split_idx:]

In [4]:
from src.logger import logging
from tensorflow.keras.layers import TextVectorization
MAX_VOCAB_SIZE = 10000 
MAX_SEQ_LEN = 64

# Initialize Keras native tokenization layer
tokenizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode='int',
    standardize='lower_and_strip_punctuation'
)

# Learn vocabulary mappings strictly from training text
tokenizer.adapt([train_text])
vocab_size = len(tokenizer.get_vocabulary())

logging.info(f"Tokenizer vocabulary built. Actual vocabulary size: {vocab_size} tokens.")

In [5]:
vocab_size

10000

In [6]:
train_ds = data_ingestion.prepare_pipeline(train_text, tokenizer)

In [7]:
val_ds = data_ingestion.prepare_pipeline(val_text, tokenizer)

In [8]:
train_ds, val_ds

(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 64), dtype=tf.int64, name=None), TensorSpec(shape=(None, 64), dtype=tf.int64, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 64), dtype=tf.int64, name=None), TensorSpec(shape=(None, 64), dtype=tf.int64, name=None))>)

In [9]:
import tensorflow as tf
import logging

# 1. Hyperparameters
D_MODEL = 128
NUM_HEADS = 4
DFF = 512
NUM_LAYERS = 2  # We will manually create a list of 2 layers
MAX_SEQ_LEN = 64
DROPOUT_RATE = 0.1

print("Initializing Embeddings and Decoder Layers directly...")

# 2. Import your individual blocks
from src.modules.embeddings.positonal_embeddings import PositionalEmbedding
from src.modules.decoder.decoder import Decoder

# 3. Instantiate the components manually
embedding_layer = PositionalEmbedding(vocab_size=vocab_size, d_model=D_MODEL, max_seq_len=MAX_SEQ_LEN)

# Create a list containing our stacked decoder layers
decoder_layers = [
    Decoder(d_model=D_MODEL, num_heads=NUM_HEADS, dff=DFF, rate=DROPOUT_RATE)
    for _ in range(NUM_LAYERS)
]

print(f"Successfully instantiated embedding layer and {NUM_LAYERS} standalone DecoderLayers.")

Initializing Embeddings and Decoder Layers directly...
Successfully instantiated embedding layer and 2 standalone DecoderLayers.


In [11]:
# Pull a single test batch from your ingestion pipeline
for test_inputs, test_targets in train_ds.take(1):
    logging.info(f"Step 0: Input token batch shape: {test_inputs.shape}") # Expected: [32, 64]
    
    # Step 1: Run through the embedding layer
    x = embedding_layer(test_inputs)
    logging.info(f"Step 1: After Sinusoidal Embedding shape: {x.shape}") # Expected: [32, 64, 128]
    
    # Step 2: Create the causal look-ahead mask for this sequence length
    seq_len = test_inputs.shape[1]
    causal_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    
    # Create a dummy encoder output matrix matching the expected shape [Batch, Seq_Len, d_model]
    # to fulfill the mandatory 'enc_output' positional parameter
    dummy_enc_output = tf.zeros_like(x)
    
    # Step 3: Pass the tensor sequentially through your stack of decoder layers
    for i, layer in enumerate(decoder_layers):
        # We explicitly map the parameters to match your class signature
        x = layer(
            x, 
            enc_output=dummy_enc_output, 
            look_ahead_mask=causal_mask, 
            padding_mask=None, 
            training=False
        )
        logging.info(f"Step 3.{i+1}: After DecoderLayer {i+1} shape: {x.shape}") # Expected: [32, 64, 128]

    print("--- Direct Decoder Verification Successful! ---")
    print(f"Final Decoder Stack output shape: {x.shape} (Matches d_model={D_MODEL})")

c:\Users\HP\Desktop\VSCode\audible\.venv\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'self_attention' (of type SelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


--- Direct Decoder Verification Successful! ---
Final Decoder Stack output shape: (32, 64, 128) (Matches d_model=128)


c:\Users\HP\Desktop\VSCode\audible\.venv\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'self_attention_2' (of type SelfAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
